# Compare Parameter Sweeps

Load saved sweep results from `data/sweeps/` and compare across instruments,
timeframes, or strategies — without re-running any backtests.

**Prerequisites:** Run a sweep in a `backtest_*.ipynb` notebook at least once.
Each sweep auto-saves to Parquet via `run_sweep()`.

**Sections:**
1. Load sweeps — all or filtered by strategy / instrument / interval
2. Summary table — best params and key stats per sweep
3. Heatmaps — side-by-side PnL% heatmaps for visual comparison
4. Parameter stability — do profitable regions overlap across sweeps?
5. Deep dive — pick any sweep and explore its full stats

In [ ]:
# ── Cell 1: Imports ────────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.backtesting import load_sweeps
from charts import plot_pnl_heatmap

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Imports OK")

In [ ]:
# ── Cell 2: Load sweeps + configure ──────────────────────────────
#
# Load everything, or filter:
#   load_sweeps(strategy="SMACross")
#   load_sweeps(instrument_id="BTC-USD-PERP.HYPERLIQUID")
#   load_sweeps(bar_interval="1h")

sweeps = load_sweeps(bar_interval="1d")

# ── CONFIGURE: your sweep parameter column names ──
# These must match the keys in your param_combos dicts.
PARAM_COLS = ["fast", "slow"]

if not sweeps:
    print("No sweeps loaded. Run a backtest notebook with run_sweep() first.")
else:
    for label, df in sweeps.items():
        swept_at = df["_swept_at"].iloc[0]
        data_start = df["_data_start"].iloc[0]
        data_end = df["_data_end"].iloc[0]
        n_bars = f"{data_start[:10]} → {data_end[:10]}"
        print(f"  {label}  ({len(df)} combos, data {n_bars})")


In [ ]:
# ── Cell 3: Best params per sweep ────────────────────────────────
#
# Ranked by PnL% (not raw PnL) so cross-asset comparison is fair
# regardless of position sizing.

if not sweeps:
    print("No sweeps loaded.")
else:
    summary_rows = []

    for label, df in sweeps.items():
        valid = df[df["total_pnl"].notna()]
        if valid.empty:
            continue

        best = valid.loc[valid["total_pnl_pct"].idxmax()]

        row = {"sweep": label}
        row["data_range"] = f"{best['_data_start'][:10]} → {best['_data_end'][:10]}"

        for pc in PARAM_COLS:
            if pc in best.index:
                row[f"best_{pc}"] = best[pc]

        row.update({
            "pnl_pct": best["total_pnl_pct"],
            "pnl": best["total_pnl"],
            "positions": int(best["num_positions"]),
        })

        # Optional analyzer stats — present if analyzer succeeded
        for stat_key, display_key in [
            ("Max Drawdown", "max_dd"),
            ("Sharpe Ratio (252 days)", "sharpe"),
            ("Profit Factor", "profit_factor"),
            ("Win Rate", "win_rate"),
            ("Expectancy", "expectancy"),
        ]:
            if stat_key in best.index:
                row[display_key] = best[stat_key]

        summary_rows.append(row)

    if summary_rows:
        summary_df = pd.DataFrame(summary_rows).sort_values("pnl_pct", ascending=False)
        print("=== Best Parameters Per Sweep (ranked by PnL%) ===")
        display(summary_df)
    else:
        print("No valid results in any loaded sweep.")


In [ ]:
# ── Cell 4: Side-by-side PnL% heatmaps ──────────────────────────
#
# Uses PnL% so heatmaps are comparable across assets with different
# position sizes and price levels.

if not sweeps:
    print("No sweeps loaded.")
else:
    ROW_COL = PARAM_COLS[1]  # y-axis (typically "slow")
    COL_COL = PARAM_COLS[0]  # x-axis (typically "fast")

    for label, df in sweeps.items():
        if ROW_COL not in df.columns or COL_COL not in df.columns:
            print(f"Skipping {label} — missing '{ROW_COL}' or '{COL_COL}' columns")
            continue
        plot_pnl_heatmap(
            df, row_col=ROW_COL, col_col=COL_COL,
            value_col="total_pnl_pct",
            row_label=f"{ROW_COL.title()} Period",
            col_label=f"{COL_COL.title()} Period",
            title=f"PnL% — {label}",
            fmt=".1f",
        )


In [ ]:
# ── Cell 5: Parameter stability ───────────────────────────────────
#
# For each param combo, average PnL% across ALL sweeps.
# Combos profitable everywhere are robust; combos that only work
# on one timeframe are likely overfit.
#
# Uses PARAM_COLS (not auto-detected) to avoid grouping by
# analyzer stats columns.

if not sweeps:
    print("No sweeps loaded.")
elif len(sweeps) < 2:
    print("Need 2+ sweeps to analyse parameter stability.")
    print("Run your backtest notebook for another instrument or timeframe,")
    print("then re-run this notebook.")
else:
    # Tag each sweep's rows with a label, then concat
    tagged = []
    for label, df in sweeps.items():
        chunk = df[PARAM_COLS + ["total_pnl", "total_pnl_pct"]].copy()
        chunk["_label"] = label
        tagged.append(chunk)

    combined = pd.concat(tagged, ignore_index=True)

    # Average PnL% across sweeps for each param combo
    stability = (
        combined
        .groupby(PARAM_COLS, as_index=False)
        .agg(
            avg_pnl_pct=("total_pnl_pct", "mean"),
            min_pnl_pct=("total_pnl_pct", "min"),
            max_pnl_pct=("total_pnl_pct", "max"),
            std_pnl_pct=("total_pnl_pct", "std"),
            sweep_count=("total_pnl_pct", "count"),
            all_profitable=("total_pnl", lambda x: (x > 0).all()),
        )
        .sort_values("avg_pnl_pct", ascending=False)
    )

    # Show combos that appear in all sweeps and are profitable everywhere
    n_sweeps = len(sweeps)
    robust = stability[
        (stability["sweep_count"] == n_sweeps) & stability["all_profitable"]
    ]

    if robust.empty:
        print(f"No param combos are profitable across all {n_sweeps} sweeps.")
        print()

        # Show how close we got — profitable in most sweeps
        mostly = stability[
            stability["sweep_count"] == n_sweeps
        ].copy()
        if not mostly.empty:
            # Count profitable sweeps per combo
            profitable_counts = (
                combined[combined["total_pnl"] > 0]
                .groupby(PARAM_COLS, as_index=False)
                .size()
                .rename(columns={"size": "n_profitable"})
            )
            mostly = mostly.merge(profitable_counts, on=PARAM_COLS, how="left")
            mostly["n_profitable"] = mostly["n_profitable"].fillna(0).astype(int)
            mostly = mostly.sort_values("n_profitable", ascending=False)

            print(f"Top 10 by profitability across {n_sweeps} sweeps:")
            display(mostly.head(10))
        else:
            print("Top 10 by average PnL% (may not appear in all sweeps):")
            display(stability.head(10))
    else:
        print(f"✅ {len(robust)} combos profitable across all {n_sweeps} sweeps:")
        display(robust.head(20))

    # Heatmap of average PnL% across all sweeps
    if len(PARAM_COLS) == 2 and len(stability) > 1:
        full_coverage = stability[stability["sweep_count"] == n_sweeps]
        if not full_coverage.empty:
            plot_pnl_heatmap(
                full_coverage,
                row_col=PARAM_COLS[1], col_col=PARAM_COLS[0],
                value_col="avg_pnl_pct",
                row_label=f"{PARAM_COLS[1].title()} Period",
                col_label=f"{PARAM_COLS[0].title()} Period",
                title=f"Avg PnL% Across {n_sweeps} Sweeps",
                fmt=".1f",
            )


In [ ]:
# ── Cell 6: Deep dive into one sweep ────────────────────────────
#
# Pick a sweep by label and explore its full stats DataFrame.

if not sweeps:
    print("No sweeps loaded.")
else:
    # Show available labels
    print("Available sweeps:")
    for i, label in enumerate(sweeps.keys()):
        print(f"  [{i}] {label}")

    # ── Change this to pick a different sweep ──
    PICK = 0

    pick_label = list(sweeps.keys())[PICK]
    pick_df = sweeps[pick_label]
    print()
    print(f"Inspecting: {pick_label}")
    print(f"Top 10 by PnL%:")

    # Only include columns that exist in this sweep
    stat_cols = [c for c in [
        "Win Rate", "Profit Factor", "Sharpe Ratio (252 days)", "Expectancy",
    ] if c in pick_df.columns]

    display(
        pick_df
        .sort_values("total_pnl_pct", ascending=False)
        .head(10)
        [PARAM_COLS + ["total_pnl", "total_pnl_pct", "num_positions"] + stat_cols]
    )
